# Sleep-EDF: исследование поставки full_psg_v1

Цель — проверить вход для будущих моделей и определить контролируемые эксперименты. Полный проход сигналов **train/val**, по одной записи в памяти. Для test — только структура, ID и заголовки массивов, без статистик сигналов и меток. Результаты описательные: обучение и подбор параметров не выполняются.

Перезапуск занимает несколько минут и читает все train/val NPZ. Спектры — воспроизводимая выборка до трёх эпох каждого класса в каждой записи (seed=42).

In [ ]:
import sys
from pathlib import Path

# На Kaggle задайте абсолютный путь к каталогу кода проекта.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (
    not (PROJECT_ROOT / "pyproject.toml").is_file()
    or not (PROJECT_ROOT / "src" / "__init__.py").is_file()
):
    raise FileNotFoundError("Укажите PROJECT_ROOT: каталог с pyproject.toml и src/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from IPython.display import display

from src.data import CHANNELS, STAGES
from src.eda import audit_dataset

DATA_ROOT = PROJECT_ROOT / "data" / "raw"  # На Kaggle замените на фактический Input.
SEED = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)


def table(frame):
    display(pd.DataFrame(frame.to_dicts()))


def plot_frame(frame):
    return pd.DataFrame(frame.to_dicts())

## Целостность и объём проверки

Проверяются SHA-256 четырёх CSV/описания по оригинальному manifest; наличие и размеры всех NPZ. Полные хэши NPZ не пересчитываются. Для всех train/val проверяются формы, частота, каналы, конечность каждого отсчёта, уникальность ID и соединение с метками по ID, записи и epoch_index. Для test массив x не распаковывается. Отсутствие пересечения людей проверяется только в размеченных train/val: идентичность людей test скрыта.

In [ ]:
audit = audit_dataset(DATA_ROOT, seed=SEED)
labels, records, quality = (audit[k] for k in ("labels", "records", "quality"))
table(
    records.group_by("split")
    .agg(
        pl.len().alias("records"),
        pl.col("epochs").sum(),
        pl.col("retained_hours").sum(),
        pl.col("gaps").sum(),
        pl.col("missing_inside").sum(),
    )
    .sort("split")
)
table(labels.group_by("split").agg(pl.col("subject_id").n_unique().alias("subjects")))
print("All configured integrity checks passed. Signal statistics cover train/val only.")

## Баланс классов и вариативность между людьми

Показываем количество и долю эпох, число людей с каждой стадией и доли стадий для каждого человека. Много эпох не равнозначно большому числу независимых наблюдений. Редкий класс может быть представлен достаточным числом эпох, но малым числом людей.

In [ ]:
balance = (
    labels.group_by("split", "label")
    .agg(pl.len().alias("epochs"), pl.col("subject_id").n_unique().alias("subjects"))
    .with_columns(
        (pl.col("epochs") / pl.col("epochs").sum().over("split")).alias("fraction")
    )
    .sort("split", "label")
)
table(balance)
sns.barplot(data=plot_frame(balance), x="label", y="fraction", hue="split")
plt.xticks(range(5), STAGES)
plt.title("Class proportions: supplied train/val")
plt.show()
subject = (
    labels.group_by("split", "subject_id", "label")
    .len()
    .with_columns(
        (pl.col("len") / pl.col("len").sum().over("split", "subject_id")).alias(
            "fraction"
        )
    )
)
wide = (
    plot_frame(subject)
    .pivot(index=["split", "subject_id"], columns="label", values="fraction")
    .fillna(0)
)
fig, ax = plt.subplots(figsize=(10, 15))
sns.heatmap(wide, ax=ax, cmap="viridis", xticklabels=STAGES, vmin=0, vmax=1)
ax.set_title("Stage fractions per subject (missing stages = 0)")
plt.tight_layout()
plt.show()

## Записи, пропуски и длительность серий

`retained_hours` — длительность сохранённых эпох, `span_hours` — расстояние между первой и последней сохранённой эпохой, а не гарантированная длительность исходного EDF. Серии стадий прерываются при пропуске epoch_index. Это наблюдаемые серии: у границ записи и пропусков они могут быть усечены.

In [ ]:
table(
    records.filter(pl.col("split") != "test")
    .select(
        "split",
        "record_id",
        "epochs",
        "retained_hours",
        "span_hours",
        "gaps",
        "missing_inside",
    )
    .sort("gaps", descending=True)
    .head(15)
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(
    data=plot_frame(records.filter(pl.col("split") != "test")),
    x="retained_hours",
    hue="split",
    bins=25,
    ax=axes[0],
)
run_summary = (
    audit["runs"]
    .group_by("split", "label")
    .agg(
        pl.len().alias("runs"),
        pl.col("epochs").median().alias("median_epochs"),
        pl.col("epochs").quantile(0.95).alias("p95_epochs"),
        pl.col("epochs").max().alias("max_epochs"),
    )
    .sort("split", "label")
)
table(run_summary)
sns.boxplot(
    data=plot_frame(audit["runs"]),
    x="label",
    y="epochs",
    hue="split",
    showfliers=False,
    ax=axes[1],
)
axes[1].set_yscale("log")
axes[1].set_xticks(range(5), STAGES)
axes[1].set_title("Observed constant-stage run length")
plt.tight_layout()
plt.show()

## Переходы и положение во времени

В переходы включены только соседние исходные эпохи одной записи; пропуски исключены. Показываем и диагональ (сохранение стадии), и условные переходы при смене стадии. Доля W во времени помогает увидеть структуру полной записи, но абсолютное время — потенциальный источник зависимости от протокола записи, не готовый признак.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for row, split in enumerate(("train", "val")):
    for col, changes_only in enumerate((False, True)):
        counts = audit["transitions"][split].copy()
        if changes_only:
            np.fill_diagonal(counts, 0)
        probs = np.divide(
            counts,
            counts.sum(axis=1, keepdims=True),
            out=np.zeros_like(counts, dtype=float),
            where=counts.sum(axis=1, keepdims=True) > 0,
        )
        sns.heatmap(
            probs,
            annot=True,
            fmt=".2f",
            vmin=0,
            vmax=1,
            xticklabels=STAGES,
            yticklabels=STAGES,
            ax=axes[row, col],
        )
        axes[row, col].set_title(
            f"{split}: {'stage changes only' if changes_only else 'all adjacent pairs'}"
        )
        axes[row, col].set(xlabel="Next", ylabel="Current")
plt.tight_layout()
plt.show()
position = labels.with_columns(
    (
        (
            pl.col("epoch_index")
            / (pl.col("epoch_index").max().over("record_id") + 1)
            * 20
        )
        .floor()
        .cast(pl.Int64)
    ).alias("bin")
)
w_position = position.group_by("split", "bin").agg(
    (pl.col("label") == 0).mean().alias("wake_fraction")
)
sns.lineplot(
    data=plot_frame(w_position), x="bin", y="wake_fraction", hue="split", marker="o"
)
plt.title("Wake fraction by relative original epoch position (pooled epochs)")
plt.show()

## Качество сигналов и амплитуды

RMS, стандартное отклонение, максимальная абсолютная амплитуда и доля одинаковых соседних отсчётов вычисляются для каждой эпохи и канала. Порог «плохого сигнала» заранее не вводится: выбросы не исключаются автоматически. Нулевое std — постоянная эпоха; повторяющиеся отсчёты могут отражать квантование, а не обязательно неисправность. Сравнение EEG и EOG проводится раздельно, в µV.

In [ ]:
quality_summary = (
    quality.group_by("split", "channel")
    .agg(
        pl.len().alias("epochs"),
        pl.col("constant").sum().alias("constant_epochs"),
        pl.col("rms_uv").median().alias("median_rms_uv"),
        pl.col("rms_uv").quantile(0.99).alias("p99_rms_uv"),
        pl.col("max_abs_uv").max().alias("peak_uv"),
        pl.col("repeated_fraction").quantile(0.99).alias("p99_repeated_fraction"),
    )
    .sort("split", "channel")
)
table(quality_summary)
table(quality.sort("max_abs_uv", descending=True).head(12))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, channel in zip(axes, CHANNELS, strict=True):
    # Только визуальная выборка; таблицы выше используют все эпохи.
    subset = quality.filter(pl.col("channel") == channel)
    sample = subset.sample(n=min(15000, subset.height), seed=SEED)
    sns.boxplot(
        data=plot_frame(sample),
        x="label",
        y="rms_uv",
        hue="split",
        showfliers=False,
        ax=ax,
    )
    ax.set_yscale("log")
    ax.set_xticks(range(5), STAGES)
    ax.set_title(channel)
plt.tight_layout()
plt.show()

## Примеры эпох и Welch PSD

Примеры взяты из train, без фильтрации и нормализации. Они иллюстрируют отдельные эпохи, не являются эталонами классов. PSD: Hann 4 с, перекрытие 50%, плотность мощности в µV²/Hz. Средние кривые усредняются сначала по выбранным эпохам внутри записи и класса, затем по записям с этим классом — это не взвешенное по всем эпохам среднее. Ресемплинга здесь нет.

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(16, 13), sharex=True)
for label, example in sorted(audit["examples"].items()):
    for channel, name in enumerate(CHANNELS):
        axes[label, channel].plot(
            np.arange(3000) / 100, example["x"][:, channel], lw=0.5
        )
        axes[label, channel].set_title(f"{STAGES[label]} · {name}")
        axes[label, channel].set_ylabel("µV")
    axes[label, 0].text(
        0, 1.22, example["id"], transform=axes[label, 0].transAxes, fontsize=8
    )
plt.tight_layout()
plt.show()
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
freq = audit["frequency"]
for row, split in enumerate(("train", "val")):
    for label in range(5):
        selected = [
            s for s in audit["spectra"] if s["split"] == split and s["label"] == label
        ]
        if not selected:
            continue
        mean_psd = np.mean([s["psd"] for s in selected], axis=0)
        for channel in range(3):
            axes[row, channel].semilogy(
                freq[1:], np.maximum(mean_psd[1:, channel], 1e-12), label=STAGES[label]
            )
            axes[row, channel].set(
                xlim=(0.25, 40),
                xlabel="Hz",
                ylabel="µV²/Hz",
                title=f"{split} · {CHANNELS[channel]}",
            )
for ax in axes.flat:
    ax.legend()
plt.tight_layout()
plt.show()
print("Sampled epochs for PSD:", sum(s["n"] for s in audit["spectra"]))

### Фактические результаты локального прохода

- Проверены все 302 653 эпохи train и 43 206 val: NaN/Inf и полностью постоянных эпох каналов не обнаружено. Это не исключает артефакты или частичное насыщение.
- Train: W 68,8%, N1 5,4%, N2 16,9%, N3 2,8%, REM 6,1%. В val N3 4,1%, REM 7,7%; состав отличается. Оценка на восьми людях val чувствительна к индивидуальным различиям.
- Разрывы исходной сетки: train 109, val 20, test 13 (для test использованы только индексы). Их нельзя игнорировать при построении последовательностей.
- Медианный RMS train: Fpz-Cz 20,6 µV, Pz-Oz 8,7 µV, EOG 51,2 µV. Общая шкала трёх каналов неравнозначна поканальной; нормализацию нужно проверять внутри обучающего разбиения.
- Выборка PSD содержит 1 865 эпох. Видимая повышенная средняя PSD REM в Pz-Oz на val требует проверки вклада отдельных записей и устойчивости к выборке; не интерпретируем её как доказанный физиологический эффект.
- Следующий эксперимент: линейная модель на абсолютных/относительных bandpower и статистиках; затем бустинг на тех же признаках и тех же группах. Контекст и веса классов проверяются отдельно.


## Выводы для экспериментов

- Сохраняем выданные группы; внутренний holdout/CV — только по subject_id внутри train. Нормализация и отбор признаков обучаются внутри fold. Val после многократного выбора решений не является независимой финальной проверкой.
- Сначала сравниваем Logistic Regression и LightGBM на одинаковых Welch/statistical features отдельных эпох; абсолютные и относительные мощности проверяем вместе. Нормализация каждой эпохи может уничтожить полезную амплитуду для N3.
- Межэпохальные признаки и BiGRU добавляем отдельным экспериментом, с разрывами и масками. Переходы стадий не дают основания автоматически сглаживать короткие N1.
- EEG/EOG и bipolar derivations сохраняем явно. Pretrained-модели получают собственный зафиксированный рецепт единиц, частоты и нормализации.
- Амплитудные выбросы сначала рассматриваем на примерах; clipping и исключение эпох требуют отдельной проверки. По этим графикам не устанавливаем универсальные пороги.
- Спектры и примеры выборочные; тестовые сигналы не использованы для оценки статистик или подбора preprocessing. Анализ не доказывает качество будущих классификаторов.

Ниже сохраняется только компактная сводка. Выполненная копия notebook хранится в artefacts/eda/, исходный notebook в Git — без outputs.

In [ ]:
import json

OUTPUT = PROJECT_ROOT / "artefacts" / "eda"
OUTPUT.mkdir(parents=True, exist_ok=True)
summary = {
    "dataset_revision": "full_psg_v1",
    "seed": SEED,
    "signal_scope": "all train/val recordings; test metadata only",
    "records": records.to_dicts(),
    "class_balance": balance.to_dicts(),
    "signal_quality": quality_summary.to_dicts(),
    "sampled_psd_epochs": sum(s["n"] for s in audit["spectra"]),
}
(OUTPUT / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

## Дополнительные проверки: люди, REM и структура записи

Статистики сигналов — только train/val. Для REM используются **все эпохи**, для остальных стадий — до 30 эпох каждой стадии каждой записи при seed 42 и 137. IQR, p95 абсолютной амплитуды и bandpower рассчитываются по эпохам первой выборки (REM целиком). Медианы по людям относятся к выборке, не ко всем эпохам остальных стадий. Повторная выборка используется для проверки устойчивости PSD. Частоты превышений EOG рассчитываются по всем эпохам. Ниже нет обучения, исключения эпох или clipping.

In [ ]:
import json

from src.data import align_labels, load_labels, load_record
from src.eda import gap_details, signal_followup, sleep_structure

followup = signal_followup(DATA_ROOT, seeds=(42, 137), per_stage=30)
FOLLOWUP_DIR = PROJECT_ROOT / "artefacts" / "eda" / "followup"
FOLLOWUP_DIR.mkdir(parents=True, exist_ok=True)
for key in ("features", "subjects", "eog"):
    followup[key].write_parquet(FOLLOWUP_DIR / f"{key}.parquet")
meta = [
    {k: v for k, v in s.items() if k not in ("mean", "median")}
    for s in followup["psds"]
]
np.savez_compressed(
    FOLLOWUP_DIR / "psd.npz",
    frequency=followup["frequency"],
    mean=np.stack([s["mean"] for s in followup["psds"]]),
    median=np.stack([s["median"] for s in followup["psds"]]),
    metadata=json.dumps(meta),
)
np.savez_compressed(
    FOLLOWUP_DIR / "examples.npz",
    **{
        f"{record}_{key}": value
        for record, example in followup["excerpts"].items()
        for key, value in example.items()
    },
)
print("Epoch-channel rows:", followup["features"].height)
print(
    "REM epochs:",
    followup["features"]
    .filter((pl.col("label") == 4) & (pl.col("channel") == CHANNELS[0]))
    .height,
)

### A. Одна стадия у разных людей

Точка — медиана эпох одного человека внутри стадии; boxplot показывает распределение этих медиан по людям, а не миллионам зависимых отсчётов. Цвет — split. Амплитуда IQR устойчива к единичным экстремальным значениям. Абсолютные мощности в µV²; относительные — к мощности 0.5–40 Hz. Для графика выбрана delta для N3 и beta для N1/REM; таблицы сохраняют все пять полос. Логарифмические оси не означают предварительную нормализацию сигнала.

In [ ]:
subject_metrics = followup["subjects"]
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for row, stage in enumerate((1, 3, 4)):
    subset = plot_frame(subject_metrics.filter(pl.col("label") == stage))
    for col, metric in enumerate(
        ("rms_uv", "iqr_uv", "relative_delta" if stage == 3 else "relative_beta")
    ):
        sns.boxplot(
            data=subset,
            x="channel",
            order=list(CHANNELS),
            hue_order=["train", "val"],
            y=metric,
            hue="split",
            showfliers=False,
            ax=axes[row, col],
        )
        sns.stripplot(
            data=subset,
            x="channel",
            order=list(CHANNELS),
            hue_order=["train", "val"],
            y=metric,
            hue="split",
            dodge=True,
            alpha=0.45,
            size=2,
            legend=False,
            ax=axes[row, col],
        )
        axes[row, col].set_title(f"{STAGES[stage]}: {metric} (subject medians)")
        axes[row, col].tick_params(axis="x", rotation=20)
        if col < 2:
            axes[row, col].set_yscale("log")
plt.tight_layout()
plt.show()
subject_range = subject_metrics.group_by("split", "label", "channel").agg(
    pl.len().alias("subjects"),
    *[
        pl.col(c).quantile(q).alias(f"{c}_p{int(q * 100)}")
        for c in ("rms_uv", "delta", "beta", "relative_beta")
        for q in (0.1, 0.5, 0.9)
    ],
)
table(subject_range.sort("label", "channel", "split"))

### B. Все REM: влияние записей и способа усреднения

Сравниваем среднее по всем REM-эпохам (вес записи пропорционален числу эпох), равный вес записей, равный вес людей, медиану средних спектров людей и иерархическую медиану: медиана эпох внутри записи → медиана ночей человека → медиана людей. Последняя устойчива к экстремальным эпохам, но это не медиана всех эпох. Дополнительно показаны записи и leave-one-subject-out: это диагностика влияния, а не исключение данных из обучения.

In [ ]:
from scipy.stats import spearmanr

psds = followup["psds"]
f = followup["frequency"]
rem = [s for s in psds if s["label"] == 4]
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
curves = {}
for ax, split in zip(axes, ("train", "val"), strict=True):
    rows = [s for s in rem if s["split"] == split]
    people = sorted({s["subject_id"] for s in rows})
    person_means = np.stack(
        [
            np.mean([s["mean"] for s in rows if s["subject_id"] == person], axis=0)
            for person in people
        ]
    )
    robust = np.stack(
        [
            np.median([s["median"] for s in rows if s["subject_id"] == person], axis=0)
            for person in people
        ]
    )
    curves[split] = {
        "epoch_mean": np.average(
            [s["mean"] for s in rows], axis=0, weights=[s["n"] for s in rows]
        ),
        "record_mean": np.mean([s["mean"] for s in rows], axis=0),
        "subject_mean": person_means.mean(axis=0),
        "subject_median": np.median(person_means, axis=0),
        "hierarchical_median": np.median(robust, axis=0),
    }
    for name, curve in curves[split].items():
        ax.semilogy(f[1:], curve[1:, 1], label=name)
    ax.set(xlim=(0.5, 40), title=f"REM Pz-Oz: {split}", xlabel="Hz", ylabel="µV²/Hz")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
rem_rows = []
for s in rem:
    band = (f >= 10) & (f < 40)
    high = np.trapezoid(s["mean"][band, 1], f[band])
    robust_high = np.trapezoid(s["median"][band, 1], f[band])
    rem_rows.append(
        {
            **{k: s[k] for k in ("split", "subject_id", "record_id", "n")},
            "power10_40": float(high),
            "integrated_median_psd10_40": float(robust_high),
        }
    )
rem_table = pl.DataFrame(rem_rows)
table(rem_table.sort("power10_40", descending=True).head(20))
fig, ax = plt.subplots(figsize=(12, 4))
for s in sorted([s for s in rem if s["split"] == "val"], key=lambda s: s["record_id"]):
    ax.semilogy(f[1:], s["mean"][1:, 1], alpha=0.7, label=s["record_id"])
ax.set(
    xlim=(0.5, 40),
    xlabel="Hz",
    ylabel="µV²/Hz",
    title="All validation recordings, REM Pz-Oz",
)
ax.legend(ncol=4, fontsize=8)
plt.show()
influence = []
for person in sorted({s["subject_id"] for s in rem if s["split"] == "val"}):
    rows = [s for s in rem if s["split"] == "val" and s["subject_id"] != person]
    curve = np.mean(
        [
            np.mean([s["mean"] for s in rows if s["subject_id"] == p], axis=0)
            for p in sorted({s["subject_id"] for s in rows})
        ],
        axis=0,
    )
    band = (f >= 10) & (f < 40)
    reference = np.trapezoid(curves["val"]["subject_mean"][band, 1], f[band])
    influence.append(
        {
            "omitted_subject": person,
            "remaining_power_ratio": float(
                np.trapezoid(curve[band, 1], f[band]) / reference
            ),
        }
    )
table(pl.DataFrame(influence).sort("remaining_power_ratio"))

Устойчивость к seed проверяется для N1/N2/N3/W по мощности 10–40 Hz внутри записи (Pz-Oz): Spearman рангов записей и медианное абсолютное изменение log10 мощности. REM не пересэмплируется, поскольку использованы все эпохи. Узкие пики ранжируются по отношению PSD к медиане соседних частот ±1 Hz (без самого бина); это исследовательская характеристика, не доказательство помех.

In [ ]:
stability = []
for stage in range(4):
    a = {s["record_id"]: s for s in psds if s["label"] == stage and s["sample"] == 0}
    b = {s["record_id"]: s for s in psds if s["label"] == stage and s["sample"] == 1}
    names = sorted(a.keys() & b.keys())
    mask = (f >= 10) & (f < 40)
    values = [
        np.array([np.trapezoid(d[n]["mean"][mask, 1], f[mask]) for n in names])
        for d in (a, b)
    ]
    stability.append(
        {
            "label": stage,
            "records": len(names),
            "spearman": float(spearmanr(*values).statistic),
            "median_abs_log10_change": float(
                np.median(
                    np.abs(
                        np.log10(
                            np.maximum(values[0], 1e-12) / np.maximum(values[1], 1e-12)
                        )
                    )
                )
            ),
        }
    )
table(pl.DataFrame(stability))
peaks = []
for s in rem:
    p = s["median"][:, 1]
    candidates = []
    for i in np.flatnonzero((f >= 5) & (f <= 39)):
        neighbours = np.r_[p[i - 4 : i], p[i + 1 : i + 5]]
        if p[i] >= p[i - 1] and p[i] >= p[i + 1]:
            candidates.append(
                (float(p[i] / max(np.median(neighbours), 1e-12)), float(f[i]))
            )
    if candidates:
        ratio, hz = max(candidates)
        peaks.append(
            {
                "record_id": s["record_id"],
                "split": s["split"],
                "peak_hz": hz,
                "local_ratio": ratio,
            }
        )
table(pl.DataFrame(peaks).sort("local_ratio", descending=True).head(12))

### C. SC4092 и экстремальные EOG

Пропуски рассматриваются по отдельности. Стадии на границах не описывают отсутствующее время. В EOG частоты >500 и >800 µV — диагностические пороги для сравнения записей, не правила очистки. Примеры содержат всю эпоху с максимальным пиком и типичную по максимальной амплитуде эпоху записи; по одному графику нельзя надёжно классифицировать физиологию/артефакт.

In [ ]:
follow_labels = load_labels(DATA_ROOT)
sc = load_record(DATA_ROOT / "epochs" / "train" / "SC4092.npz", signals=False)
sy = align_labels(sc, follow_labels, "SC4092", "train")
gaps, segments = gap_details(sc["epoch_index"], sy)
table(gaps)
table(segments)
fig, ax = plt.subplots(figsize=(12, 2))
for row in segments.iter_rows(named=True):
    ax.plot([row["first_index"] / 120, (row["last_index"] + 1) / 120], [1, 1], lw=8)
ax.set(
    xlabel="Hours from original PSG start", yticks=[], title="SC4092 retained segments"
)
plt.show()
eog = followup["eog"].with_columns(
    (pl.col("over500") / pl.col("epochs")).alias("fraction500"),
    (pl.col("over800") / pl.col("epochs")).alias("fraction800"),
)
table(
    eog.filter(pl.col("record_id").is_in(["SC4242", "SC4592"])).sort(
        "record_id", "label"
    )
)
eog_records = (
    eog.group_by("split", "record_id", "subject_id")
    .agg(
        pl.col("epochs").sum(),
        pl.col("over500").sum(),
        pl.col("over800").sum(),
        pl.col("max_uv").max(),
    )
    .with_columns((pl.col("over800") / pl.col("epochs")).alias("fraction800"))
)
table(eog_records.sort("fraction800", descending=True).head(15))
fig, axes = plt.subplots(2, 2, figsize=(15, 7))
for row, (name, example) in enumerate(followup["excerpts"].items()):
    for col in range(2):
        axes[row, col].plot(np.arange(3000) / 100, example["x"][col, :, 2], lw=0.7)
        axes[row, col].set(
            title=f"{example['ids'][col]} · {STAGES[int(example['labels'][col])]} · peak {example['peaks'][col]:.0f} µV",
            xlabel="s",
            ylabel="EOG µV",
        )
plt.tight_layout()
plt.show()

### D. Крайние W и диагностический период сна

Диагностическое ядро — от первой до последней сохранённой non-W эпохи включительно, **без добавочного контекста**. Пропуски остаются пропусками. Используются метки train/val; этот способ нельзя применять как доступный на test preprocessing или как безусловную обрезку обучения. Профиль положения считается отдельно по людям; сравнение начала/центра/конца использует фиксированные 20%/60%/20% исходной сетки.

In [ ]:
structure = sleep_structure(follow_labels)
table(structure["record_wake"].sort("wake_fraction", descending=True).head(15))
table(structure["class_scope"].sort("split", "scope", "label"))
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
sns.boxplot(
    data=plot_frame(structure["record_wake"]), x="split", y="wake_fraction", ax=axes[0]
)
people = follow_labels.group_by("split", "subject_id").agg(
    (pl.col("label") == 0).mean().alias("wake_fraction")
)
sns.stripplot(
    data=plot_frame(people),
    x="split",
    y="wake_fraction",
    ax=axes[0],
    color="black",
    size=3,
)
axes[0].set_title("Boxes: records; dots: subjects")
sns.boxplot(
    data=plot_frame(structure["regions"]),
    x="region",
    y="wake_fraction",
    hue="split",
    ax=axes[1],
)
profile = plot_frame(structure["position"])
for split, color in (("train", "C0"), ("val", "C1")):
    pivot = profile[profile["split"] == split].pivot(
        index="subject_id", columns="position_bin", values="wake_fraction"
    )
    for _, row in pivot.iterrows():
        axes[2].plot(pivot.columns, row, color=color, alpha=0.1)
    axes[2].plot(pivot.columns, pivot.median(), color=color, label=split)
axes[2].set(
    title="Individual profiles + median of subjects",
    xlabel="Position bin",
    ylabel="Wake fraction",
)
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
for name, frame in structure.items():
    frame.write_parquet(FOLLOWUP_DIR / f"{name}.parquet")
rem_table.write_parquet(FOLLOWUP_DIR / "rem_records.parquet")
pl.DataFrame(influence).write_parquet(FOLLOWUP_DIR / "rem_influence.parquet")
pl.DataFrame(stability).write_parquet(FOLLOWUP_DIR / "sampling_stability.parquet")
pl.DataFrame(peaks).write_parquet(FOLLOWUP_DIR / "narrow_peaks.parquet")
gaps.write_parquet(FOLLOWUP_DIR / "sc4092_gaps.parquet")
segments.write_parquet(FOLLOWUP_DIR / "sc4092_segments.parquet")
(FOLLOWUP_DIR / "method.json").write_text(
    json.dumps(
        {
            "revision": "full_psg_v1",
            "seeds": [42, 137],
            "per_stage_record": 30,
            "rem": "all epochs",
            "scope": "train/val only",
        },
        indent=2,
    ),
    encoding="utf-8",
)

### Адресная проверка источников спектральных отличий

SC4261 и SC4172 выделились по среднему REM PSD. Проверяем распределение мощности 10–40 Hz по всем REM-эпохам, долю мощности верхних 5% эпох и примеры типичной/экстремальной эпохи. Для узких пиков SC4272 и SC4501 сравниваем каналы и стадии: все REM и до 100 равномерно расположенных эпох каждой другой стадии. Это отдельный диагностический срез, не правило очистки.

In [ ]:
from src.eda import inspect_spectral_records

spectral_check = inspect_spectral_records(DATA_ROOT)
table(spectral_check["details"])
table(
    spectral_check["peak_controls"].filter(
        pl.col("record_id").is_in(["SC4272", "SC4501"])
    )
)
spectral_check["details"].write_parquet(FOLLOWUP_DIR / "rem_epoch_tails.parquet")
spectral_check["peak_controls"].write_parquet(FOLLOWUP_DIR / "peak_controls.parquet")
fig, axes = plt.subplots(2, 3, figsize=(17, 7))
for row, (record_id, example) in enumerate(spectral_check["examples"].items()):
    for col in range(2):
        axes[row, col].plot(np.arange(3000) / 100, example["x"][col, :, 1], lw=0.6)
        axes[row, col].set(
            title=f"{example['ids'][col]} · power10–40={example['power'][col]:.2f} µV²",
            xlabel="s",
            ylabel="Pz-Oz µV",
        )
        axes[row, 2].semilogy(
            spectral_check["frequency"][1:],
            example["psd"][col, 1:],
            label=("typical" if col == 0 else "extreme"),
        )
    axes[row, 2].set(
        title=f"{record_id}: REM", xlim=(0.5, 40), xlabel="Hz", ylabel="µV²/Hz"
    )
    axes[row, 2].legend()
plt.tight_layout()
plt.show()

## Итог дополнительных проверок

**1. Что подтвердилось.** Межличностный разброс сигналов существует внутри одинаковой стадии, а не только в долях классов. Например, для N1 Fpz-Cz 10–90-й процентили медиан RMS людей train — примерно 8,0–14,9 µV; для N3 — 18,3–30,3 µV. По большинству рассмотренных медианных характеристик train/val существенно перекрываются. Выборочные оценки стадий, кроме REM, не заменяют полные распределения. Для этих стадий смена seed сохраняет ранги записей по мощности 10–40 Hz Pz-Oz: Spearman 0,96–0,99.

**2. Что объясняется отдельными записями и агрегированием.** Все 21 823 REM-эпохи использованы, поэтому малая исходная PSD-выборка больше не ограничивает эту проверку. SC4261: средняя мощность Pz-Oz 10–40 Hz 28,45 µV² против медианы мощности эпох 1,47 µV²; верхние 5% эпох дают 92,8% суммарной мощности. SC4172: средняя 16,88, медиана 3,85 µV². Примеры имеют резкие широкополосные отклонения, совместимые с загрязнением/артефактами; источник по NPZ достоверно не устанавливается. Исключение SC426 только из диагностического среднего по людям снижает его на 24,1%; обучающие данные не исключались.

Среднее по людям 10–40 Hz Pz-Oz REM: train 3,60, val 5,75 µV². После иерархических медиан (эпохи → ночи → люди) — 2,59 и 2,50 µV². **Интеграл медианного PSD и медиана интегралов PSD эпох — разные величины**, обе подписаны явно. Пик около 26,75 Hz в SC4272 и 27 Hz в SC4501 сохраняется на нескольких стадиях, сильнее в Pz-Oz; он встречается и в train, и в val. Это скорее кандидат на особенность записи/помеху, чем специфический REM-маркер; причину не доказываем и notch автоматически не вводим.

**3. Подозрительные записи.** SC4092: один пропуск 802 эпохи (6 ч 41 мин) начинается на 17,125 ч исходной сетки; до/после W. Остальные 12 пропусков — по одной эпохе. Получается 14 непрерывных сегментов; содержание отсутствующего времени неизвестно.

SC4242: 1186/2710 эпох имеют EOG peak >500 µV (43,8%), главным образом W — 1148/1879; >800 µV только 11 эпох. SC4592: >500 µV 36/2040 (1,8%), >800 µV 4/2040. В показанных фрагментах есть большие движения/скачки; отличить физиологию от технических артефактов надёжно нельзя. В train также есть записи с частыми высокими EOG-амплитудами — это не исключительно свойство val. Пороги диагностические, не правила очистки.

**4. Положение и длинные W.** После диагностического удаления только крайних W доля W падает с 68,8% до 28,1% в train и с 66,6% до 24,5% в val. N3 становится 6,4% и 9,3%. Все non-W эпохи при этом сохранены по определению; меняется знаменатель и состав W. Профили людей подтверждают влияние структуры записи, но не универсальный закон сна. Такая граница определяется метками и не является допустимым доступным на test способом обрезки. Ничего не удаляем из обучения.

**5. Решение для baseline и ограничения.** Есть индивидуальные различия и отдельные загрязнённые эпохи/узкие пики. Убедительного подтверждения общего сдвига типичных EEG/EOG-характеристик всего val на этих проверках нет; отсутствие сдвига не доказано, val содержит лишь 8 людей. Не сравниваем миллионы отсчётов как независимые наблюдения.

Для Logistic Regression берём log абсолютных bandpower, относительные мощности/отношения полос, RMS, IQR и p95 абсолютной амплитуды с поканальным учётом масштаба. Scaler обучаем только внутри train-fold. Затем сравниваем LightGBM/CatBoost на тех же признаках. Начинаем без position features, обрезки W, удаления людей, clipping и notch; каждое такое преобразование требует отдельного эксперимента. Контекст строится по непрерывным сегментам. Открыты точное происхождение артефактов/пиков и влияние робастной обработки на качество классификации — это уже проверяется моделями.

Дополнительные секции выполнены отдельно; ранее выполненные таблицы и графики EDA сохранены. Сигналы test не использовались, обучение не запускалось.
